# 한국어 대화 요약 - Qwen3-14B LoRA SFT (Unsloth)

## 개요
이 노트북은 **Qwen3-14B** 모델을 **LoRA (Low-Rank Adaptation)** 방식으로 파인튜닝하여 한국어 대화를 요약하는 방법을 다룹니다.

### 핵심 기술 스택
- **Unsloth**: 메모리 효율적인 LLM 학습 라이브러리 (4-bit 양자화 지원)
- **LoRA**: 전체 모델 파라미터 중 약 0.86%만 학습하여 GPU 메모리 절약
- **Response-only Loss**: 프롬프트 부분은 학습하지 않고, 모델의 응답(요약)만 학습

### 실험 결과 요약

| 방법 | Mecab ROUGE-1 | Mecab ROUGE-2 | 비고 |
|------|--------------|--------------|------|
| KoBART baseline | ~0.35 | - | 대회 제공 baseline |
| Qwen3-14B LoRA SFT (이 코드) | **0.5641** | **0.3849** | Best 단일 모델 |
| MBR 8-model 앙상블 | **0.5716** | **0.3883** | Best 전체 |
| 리더보드 타겟 | 0.5600 | 0.3640 | 대회 목표 |

### GPU 요구사항
- **RTX 3090 (24GB)**: `batch_size=1`, `grad_accum=32` 로 설정
- **RTX A6000 (48GB)**: `batch_size=4`, `grad_accum=8` 로 설정
- **A100 (80GB)**: `batch_size=8`, `grad_accum=4` 로 설정

> 4-bit 양자화 덕분에 14B 모델도 24GB GPU에서 학습 가능합니다!

## 1. 환경 설정

필요한 패키지를 설치합니다. Unsloth는 PyTorch와 CUDA가 사전 설치되어 있어야 합니다.

In [ ]:
# 패키지 설치 (처음 한 번만 실행)
# !pip install unsloth peft trl rouge datasets pandas wandb mecab-python3

In [ ]:
import os
import re
import random
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from rouge import Rouge
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import Any, Dict, List

# 재현성을 위한 시드 고정
random.seed(3407)
np.random.seed(3407)
torch.manual_seed(3407)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. 하이퍼파라미터 설정

GPU 메모리에 따라 배치 사이즈를 조정합니다.

**핵심 원리**: `effective_batch_size = per_device_batch_size × gradient_accumulation_steps`
- RTX 3090에서는 배치 1개씩만 넣을 수 있지만, gradient accumulation으로 32개를 모아서 한 번에 업데이트합니다.
- 결과적으로 배치 4 × accumulation 8 = 배치 1 × accumulation 32 = 동일한 학습 효과!

In [ ]:
# ============================================================
# 하이퍼파라미터 (RTX 3090 24GB 기준)
# ============================================================

# 실험 이름 (결과 저장 폴더명)
EXP_NAME = "qwen3_14b_lora_sft"

# 모델 설정
MODEL_NAME = "unsloth/Qwen3-14B"  # Unsloth 최적화된 Qwen3-14B
MAX_SEQ_LENGTH = 2048              # 최대 시퀀스 길이

# LoRA 설정
LORA_R = 32          # LoRA rank (높을수록 표현력 ↑, 메모리 ↑)
LORA_ALPHA = 32      # LoRA scaling factor
LORA_DROPOUT = 0.0   # LoRA dropout (Unsloth에서는 0 권장)

# 학습 설정
LEARNING_RATE = 2e-4  # 학습률
EPOCHS = 3            # 에폭 수
WARMUP_RATIO = 0.05   # 워밍업 비율 (전체 스텝의 5%)
WEIGHT_DECAY = 0.01   # 가중치 감쇠

# ★ GPU 메모리에 따라 이 두 값을 조정하세요!
# RTX 3090 (24GB): batch=1, accum=32
# RTX A6000 (48GB): batch=4, accum=8
# A100 (80GB): batch=8, accum=4
PER_DEVICE_BATCH = 1   # GPU 한 장에 넣는 배치 수
GRAD_ACCUM = 32        # gradient accumulation steps

# 추론 설정
MAX_NEW_TOKENS = 192   # 생성할 최대 토큰 수

print(f"Effective batch size: {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 3. 데이터 로드 및 탐색

대회 데이터는 `train.csv`, `dev.csv`, `test.csv` 세 파일로 구성됩니다.
- `dialogue`: 대화 원문 (화자 태그 #Person1#, #Person2# 등 포함)
- `summary`: 정답 요약 (train/dev에만 존재)
- `topic`: 대화 주제
- `fname`: 파일명 (제출용)

In [ ]:
# 데이터 로드
DATA_PATH = "./data"  # 데이터 경로를 맞게 수정하세요

train_df = pd.read_csv(os.path.join(DATA_PATH, "train.csv"))
dev_df = pd.read_csv(os.path.join(DATA_PATH, "dev.csv"))
test_df = pd.read_csv(os.path.join(DATA_PATH, "test.csv"))

print(f"Train: {len(train_df):,}개")
print(f"Dev:   {len(dev_df):,}개")
print(f"Test:  {len(test_df):,}개")
print(f"\n컬럼: {train_df.columns.tolist()}")

In [ ]:
# 데이터 탐색
print("=" * 60)
print("[예시 1] 대화:")
print(train_df.iloc[0]["dialogue"][:500])
print(f"\n[예시 1] 요약:")
print(train_df.iloc[0]["summary"])
print("=" * 60)

In [ ]:
# 데이터 통계 분석
print("[대화 길이 통계]")
print(f"  평균: {train_df['dialogue'].str.len().mean():.0f}자")
print(f"  최소: {train_df['dialogue'].str.len().min()}자")
print(f"  최대: {train_df['dialogue'].str.len().max()}자")

print(f"\n[요약 길이 통계]")
print(f"  평균: {train_df['summary'].str.len().mean():.0f}자")
print(f"  최소: {train_df['summary'].str.len().min()}자")
print(f"  최대: {train_df['summary'].str.len().max()}자")

# #PersonN# 태그 사용 비율
has_person_tag = train_df['summary'].str.contains(r'#Person\d+#', regex=True).mean()
print(f"\n요약에 #PersonN# 태그 포함 비율: {has_person_tag:.1%}")

## 4. 모델 로드 (4-bit 양자화)

Unsloth를 사용하면 14B 파라미터 모델을 4-bit로 양자화하여 로드할 수 있습니다.
- **원본 모델 크기**: ~28GB (FP16)
- **4-bit 양자화 후**: ~7GB
- 나머지 VRAM은 학습에 사용됩니다.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,   # 4-bit 양자화 (메모리 절약의 핵심!)
    load_in_8bit=False,
    full_finetuning=False,
)

print(f"모델 로드 완료: {MODEL_NAME}")

## 5. LoRA 어댑터 설정

**LoRA (Low-Rank Adaptation)** 이란?
- 전체 모델의 가중치를 직접 수정하지 않고, 작은 행렬(어댑터)만 추가하여 학습합니다.
- 14B 파라미터 중 약 **1.28억 개 (0.86%)** 만 학습합니다.
- 학습이 빠르고, GPU 메모리를 적게 사용합니다.

### LoRA 핵심 파라미터
- `r` (rank): 어댑터의 크기. 클수록 표현력이 높지만 메모리를 더 사용합니다.
- `lora_alpha`: 스케일링 팩터. 보통 `r`과 같은 값으로 설정합니다.
- `target_modules`: LoRA를 적용할 레이어. Transformer의 attention과 FFN 레이어에 적용합니다.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,                      # LoRA rank
    lora_alpha=LORA_ALPHA,         # 스케일링 팩터
    lora_dropout=LORA_DROPOUT,     # 드롭아웃
    bias="none",                   # bias는 학습하지 않음
    use_gradient_checkpointing="unsloth",  # 메모리 최적화
    random_state=3407,
    use_rslora=False,              # Rank-Stabilized LoRA (선택사항)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention 레이어
        "gate_proj", "up_proj", "down_proj",      # FFN 레이어
    ],
)

# 학습 가능한 파라미터 수 확인
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"학습 가능 파라미터: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 6. 프롬프트 설계

LLM은 **채팅 형식(Chat Template)** 으로 입력을 받습니다.
- `system`: 모델에게 역할을 부여합니다 ("대화 요약 전문가")
- `user`: 사용자의 요청 (대화 원문 포함)
- `assistant`: 모델의 응답 (요약, 학습 시에만 사용)

### 프롬프트 설계 팁
1. `#Person1#`, `#Person2#` 등 화자 태그를 유지하라고 명시
2. 요약 길이를 "1~3문장"으로 제한
3. 핵심 내용만 포함하라고 지시

In [ ]:
# 프롬프트 설정
SYSTEM_PROMPT = (
    "당신은 한국어 대화 요약 전문가입니다. "
    "대화에는 #Person1#, #Person2# 등의 화자 태그가 사용됩니다. "
    "요약할 때 이 화자 태그를 그대로 사용하여 누가 무엇을 했는지 명확히 구분해주세요. "
    "핵심 내용만 1~3문장으로 간결하게 요약하세요."
)

USER_TEMPLATE = (
    "아래 대화를 읽고 핵심 내용을 요약해주세요. "
    "화자 태그(#Person1# 등)를 유지하세요.\n\n{dialogue}"
)


def create_messages(dialogue, summary=None, is_train=True):
    """채팅 형식의 메시지 리스트를 생성합니다.
    
    Args:
        dialogue: 대화 원문
        summary: 정답 요약 (학습 시에만 사용)
        is_train: 학습 모드 여부
    
    Returns:
        messages: [{role, content}, ...] 형태의 리스트
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_TEMPLATE.format(dialogue=dialogue)},
    ]
    # 학습 시에는 정답 요약을 assistant 응답으로 추가
    if is_train and summary is not None:
        messages.append({"role": "assistant", "content": str(summary)})
    return messages


# 프롬프트 예시 확인
sample_messages = create_messages(
    train_df.iloc[0]["dialogue"], 
    train_df.iloc[0]["summary"]
)
sample_text = tokenizer.apply_chat_template(
    sample_messages, tokenize=False, 
    add_generation_prompt=False, enable_thinking=False
)
print("[채팅 템플릿 적용 결과 (앞 500자)]")
print(sample_text[:500])
print("...")
print(sample_text[-200:])

## 7. Response-Only Loss 설정

### 왜 Response-Only Loss가 중요한가?

일반적인 SFT(Supervised Fine-Tuning)에서는 전체 텍스트(시스템 프롬프트 + 사용자 입력 + 모델 응답)의 모든 토큰에 대해 loss를 계산합니다.

하지만 우리가 학습하고 싶은 것은 **모델의 응답(요약)** 뿐입니다!

```
[시스템 프롬프트] → loss 계산 X (마스킹)
[사용자 입력]     → loss 계산 X (마스킹)
[모델 응답]       → loss 계산 O ← 이것만 학습!
```

이렇게 하면:
- 모델이 프롬프트를 외우는 대신 **요약 생성에 집중**합니다
- 같은 학습 시간에 더 좋은 성능을 얻을 수 있습니다

In [ ]:
@dataclass
class ResponseOnlyDataCollator:
    """프롬프트 토큰을 -100으로 마스킹하여 응답(요약)만 학습하는 DataCollator
    
    PyTorch의 CrossEntropyLoss는 label이 -100인 토큰을 무시합니다.
    이를 이용해 프롬프트 부분의 label을 -100으로 설정하면,
    모델 응답 부분에 대해서만 loss가 계산됩니다.
    """
    tokenizer: Any
    response_template_ids: List[int] = None  # assistant 응답 시작 토큰 ID
    max_length: int = 2048

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        # SFTTrainer가 이미 토크나이즈한 경우
        if "input_ids" in features[0]:
            batch = self.tokenizer.pad(
                features, padding=True, 
                max_length=self.max_length, return_tensors="pt"
            )
        else:
            # raw text인 경우 직접 토크나이즈
            texts = [f["text"] for f in features]
            batch = self.tokenizer(
                texts, return_tensors="pt", padding=True,
                truncation=True, max_length=self.max_length
            )

        labels = batch["input_ids"].clone()

        # 각 샘플에서 assistant 응답 시작점을 찾아 그 이전을 마스킹
        for i in range(len(labels)):
            ids = batch["input_ids"][i].tolist()
            response_start = self._find_response_start(ids)
            if response_start >= 0:
                labels[i, :response_start] = -100  # 프롬프트 부분 마스킹
            # 패딩 토큰도 마스킹
            labels[i, batch["attention_mask"][i] == 0] = -100

        batch["labels"] = labels
        return batch

    def _find_response_start(self, ids: List[int]) -> int:
        """토큰 시퀀스에서 assistant 응답 시작 위치를 찾습니다."""
        template = self.response_template_ids
        if template is None:
            return 0
        last_pos = -1
        for i in range(len(ids) - len(template) + 1):
            if ids[i:i + len(template)] == template:
                last_pos = i + len(template)
        return last_pos

In [ ]:
# Response template 설정
# Qwen3 모델은 assistant 응답 시작을 "<|im_start|>assistant\n" 으로 표시합니다.

response_template_str = "<|im_start|>assistant\n"
response_template_ids = tokenizer.encode(response_template_str, add_special_tokens=False)

print(f"Response template: {repr(response_template_str)}")
print(f"Response template IDs: {response_template_ids}")

# 검증: 실제 학습 데이터에서 template이 찾아지는지 확인
sample_ids = tokenizer.encode(sample_text, add_special_tokens=False)
collator = ResponseOnlyDataCollator(
    tokenizer=tokenizer,
    response_template_ids=response_template_ids,
    max_length=MAX_SEQ_LENGTH,
)
pos = collator._find_response_start(sample_ids)
print(f"\n검증: {pos}/{len(sample_ids)} 토큰 마스킹 ({pos/len(sample_ids)*100:.1f}%)")
print("→ 프롬프트 부분은 학습에서 제외됩니다!")

## 8. 학습 데이터 준비

각 대화-요약 쌍을 채팅 템플릿 형식의 텍스트로 변환합니다.

In [ ]:
def formatting_prompts_func(examples):
    """대화-요약 쌍을 채팅 템플릿 텍스트로 변환합니다."""
    texts = []
    for dialogue, summary in zip(examples["dialogue"], examples["summary"]):
        messages = create_messages(dialogue, summary, is_train=True)
        text = tokenizer.apply_chat_template(
            messages, tokenize=False,
            add_generation_prompt=False, enable_thinking=False,
        )
        texts.append(text)
    return {"text": texts}


# 데이터셋 변환
train_dataset = Dataset.from_pandas(train_df[["dialogue", "summary"]]).map(
    formatting_prompts_func, batched=True
)
dev_dataset = Dataset.from_pandas(dev_df[["dialogue", "summary"]]).map(
    formatting_prompts_func, batched=True
)

print(f"학습 데이터: {len(train_dataset):,}개")
print(f"검증 데이터: {len(dev_dataset):,}개")
print(f"\n변환된 텍스트 예시 (앞 200자):")
print(train_dataset[0]["text"][:200])

## 9. 학습 (SFT)

Hugging Face의 `SFTTrainer`를 사용하여 학습합니다.

### 학습 설정 설명
- `cosine` 스케줄러: 학습률이 코사인 곡선을 따라 감소합니다
- `adamw_8bit`: 8-bit AdamW 옵티마이저 (메모리 절약)
- `bf16`: BFloat16 혼합 정밀도 학습 (A100/3090 지원)
- `gradient_checkpointing`: 중간 활성화값을 저장하지 않고 재계산 (메모리 ↓, 속도 약간 ↓)

### 예상 학습 시간
- RTX 3090: 약 4-5시간
- RTX A6000: 약 2-3시간
- A100: 약 1-2시간

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

OUTPUT_DIR = f"./outputs/{EXP_NAME}"

sft_config = SFTConfig(
    # 데이터 설정
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,  # 여러 샘플을 하나로 묶지 않음
    
    # 배치 설정 (★ GPU에 따라 조정)
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    
    # 학습률 설정
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    
    # 정밀도 설정
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    
    # 로깅 및 저장
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    
    # 기타
    seed=3407,
    output_dir=OUTPUT_DIR,
    report_to="none",  # wandb 사용 시 "wandb"로 변경
    optim="adamw_8bit",
    max_grad_norm=1.0,
)

# Response-only DataCollator 적용
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    args=sft_config,
    data_collator=collator,  # ← Response-only loss의 핵심!
)

print(f"학습 준비 완료!")
print(f"  - Effective batch size: {PER_DEVICE_BATCH * GRAD_ACCUM}")
print(f"  - 총 스텝 수: {len(train_dataset) // (PER_DEVICE_BATCH * GRAD_ACCUM) * EPOCHS}")
print(f"  - BF16: {is_bfloat16_supported()}")

In [ ]:
# ★ 학습 시작!
print(f"[{EXP_NAME}] 학습 시작...")
trainer_stats = trainer.train()

print(f"\n학습 완료!")
print(f"  총 학습 시간: {trainer_stats.metrics['train_runtime']:.0f}초")
print(f"  최종 train loss: {trainer_stats.metrics['train_loss']:.4f}")

In [ ]:
# LoRA 어댑터 저장
LORA_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(LORA_PATH)
tokenizer.save_pretrained(LORA_PATH)
print(f"LoRA 어댑터 저장 완료: {LORA_PATH}")

## 10. 추론 (Inference)

학습이 끝나면 모델을 추론 모드로 전환하고 요약을 생성합니다.

### 후처리 (Postprocessing)
생성된 요약에서 불필요한 태그나 형식을 정리합니다:
1. `<think>...</think>` 태그 제거 (Chain-of-Thought 잔여물)
2. HTML/특수 태그 제거
3. `#Person N#` → `#PersonN#` 정규화
4. 중복 공백 정리

In [ ]:
# 추론 모드 전환
FastLanguageModel.for_inference(model)


def postprocess(text):
    """생성된 요약을 후처리합니다."""
    # think 태그 제거
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # HTML/특수 태그 제거
    text = re.sub(r"<[^>]+>", "", text)
    # #Person N# → #PersonN# 정규화
    text = re.sub(r"#\s*Person\s*(\d+)\s*#", r"#Person\1#", text)
    # 연속 공백 정리
    text = re.sub(r"\s+", " ", text).strip()
    # "요약:" 접두사 제거
    text = re.sub(r"^요약\s*:\s*", "", text).strip()
    return text if text else "빈 요약"


def generate_summary(dialogue):
    """대화 원문으로부터 요약을 생성합니다."""
    messages = create_messages(dialogue, is_train=False)
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=True,  # assistant 응답 시작 토큰 추가
        enable_thinking=False,       # Chain-of-Thought 비활성화
    )
    inputs = tokenizer(
        text, return_tensors="pt", 
        truncation=True, max_length=MAX_SEQ_LENGTH
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,  # Greedy decoding (재현성 보장)
        )
    
    # 입력 부분을 제외한 생성된 토큰만 디코딩
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    summary = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return postprocess(summary)

In [ ]:
# 추론 예시 확인 (dev 3개)
print("[추론 예시]")
for i in range(3):
    pred = generate_summary(dev_df.iloc[i]["dialogue"])
    gold = dev_df.iloc[i]["summary"]
    print(f"\n--- 예시 {i+1} ---")
    print(f"  정답: {gold}")
    print(f"  예측: {pred}")

## 11. Dev Set 평가 (ROUGE)

ROUGE 점수로 모델 성능을 평가합니다.
- **ROUGE-1**: 1-gram(단어) 단위 겹침
- **ROUGE-2**: 2-gram(연속 2단어) 단위 겹침  
- **ROUGE-L**: 최장 공통 부분 수열 기반

> 대회에서는 **Mecab 형태소 분석기**로 토큰화한 후 ROUGE를 계산합니다.

In [ ]:
# Dev set 전체 추론
print(f"Dev set 추론 시작 ({len(dev_df)}개)...")
dev_preds = []
for i in tqdm(range(len(dev_df))):
    pred = generate_summary(dev_df.iloc[i]["dialogue"])
    dev_preds.append(pred)

print(f"추론 완료: {len(dev_preds)}개")

In [ ]:
# ROUGE 평가 (공백 기반)
rouge = Rouge()
golds = [str(s).strip() if str(s).strip() else "빈 요약" for s in dev_df["summary"]]
preds = [p if p.strip() else "빈 요약" for p in dev_preds]

scores = rouge.get_scores(preds, golds, avg=True)
print("=" * 50)
print(f"[{EXP_NAME}] Dev ROUGE (공백 기반):")
print(f"  ROUGE-1 F1: {scores['rouge-1']['f']:.4f}")
print(f"  ROUGE-2 F1: {scores['rouge-2']['f']:.4f}")
print(f"  ROUGE-L F1: {scores['rouge-l']['f']:.4f}")
print("=" * 50)

In [ ]:
# ROUGE 평가 (Mecab 형태소 기반 - 대회 공식 기준)
try:
    import mecab
    m = mecab.MeCab()
    
    golds_mecab = [" ".join(m.morphs(g)) for g in golds]
    preds_mecab = [" ".join(m.morphs(p)) for p in preds]
    
    mecab_scores = rouge.get_scores(preds_mecab, golds_mecab, avg=True)
    print("=" * 50)
    print(f"[{EXP_NAME}] Dev ROUGE (Mecab 형태소 기반, 대회 공식):")
    print(f"  ROUGE-1 F1: {mecab_scores['rouge-1']['f']:.4f}")
    print(f"  ROUGE-2 F1: {mecab_scores['rouge-2']['f']:.4f}")
    print(f"  ROUGE-L F1: {mecab_scores['rouge-l']['f']:.4f}")
    print("=" * 50)
    print(f"\n참고: 리더보드 타겟 ROUGE-1=0.5600, ROUGE-2=0.3640")
except ImportError:
    print("mecab 패키지가 설치되지 않았습니다.")
    print("pip install mecab-python3 으로 설치할 수 있습니다.")
    print("(공백 기반 ROUGE만 표시됩니다)")

## 12. Test Set 추론 및 제출 파일 생성

Test set에 대해 요약을 생성하고, 대회 제출 형식의 CSV 파일을 만듭니다.

제출 파일 형식:
```
fname,summary
test_0,#Person1#이 #Person2#에게 ...
test_1,...
```

In [ ]:
# Test set 추론
print(f"Test set 추론 시작 ({len(test_df)}개)...")
test_preds = []
for i in tqdm(range(len(test_df))):
    pred = generate_summary(test_df.iloc[i]["dialogue"])
    test_preds.append(pred)

print(f"추론 완료: {len(test_preds)}개")

In [ ]:
# 제출 파일 생성
os.makedirs("prediction", exist_ok=True)

submission = pd.DataFrame({
    "fname": test_df["fname"],
    "summary": test_preds,
})

sub_path = f"prediction/submission_{EXP_NAME}.csv"
submission.to_csv(sub_path, index=False)

print(f"제출 파일 저장 완료: {sub_path}")
print(f"총 {len(submission)}개 행")
print(f"\n[제출 파일 미리보기]")
print(submission.head())

In [ ]:
# 생성된 요약 통계
print(f"[생성된 요약 통계]")
print(f"  평균 길이: {submission['summary'].str.len().mean():.0f}자")
print(f"  최소 길이: {submission['summary'].str.len().min()}자")
print(f"  최대 길이: {submission['summary'].str.len().max()}자")
print(f"  #PersonN# 포함 비율: {submission['summary'].str.contains('#Person', regex=False).mean():.1%}")
print(f"\n[예시 3개]")
for i in range(3):
    print(f"  {submission.iloc[i]['fname']}: {submission.iloc[i]['summary'][:100]}")

## 13. 저장된 LoRA 어댑터 로드 (나중에 재사용)

학습이 완료된 후, 저장된 LoRA 어댑터를 다시 로드하여 추론에 사용할 수 있습니다.

```python
# 새 세션에서 모델 + LoRA 로드
from unsloth import FastLanguageModel
from peft import PeftModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen3-14B",
    max_seq_length=2048,
    load_in_4bit=True,
)
model = PeftModel.from_pretrained(model, "outputs/qwen3_14b_lora_sft/lora_adapter")
FastLanguageModel.for_inference(model)
```

## 부록: 성능 향상 분석

### 실험 결과 비교

| 방법 | 모델 | MC-R1 | MC-R2 | KoBART 대비 |
|------|------|-------|-------|------------|
| KoBART baseline | KoBART | ~0.35 | - | 기준 |
| Qwen3-14B LoRA SFT (이 코드) | Qwen3-14B | 0.5641 | 0.3849 | +0.21 |
| + 프롬프트 변형 (추상적) | Qwen3-14B | 0.5575 | 0.3768 | +0.21 |
| + 프롬프트 변형 (goldstyle) | Qwen3-14B | 0.5573 | 0.3736 | +0.21 |
| Qwen3-32B LoRA SFT | Qwen3-32B | 0.5433 | 0.3627 | +0.19 |
| MBR 8-model 앙상블 | 여러 모델 | **0.5716** | **0.3883** | **+0.22** |

### 성능 향상의 핵심 요인

1. **큰 모델 (14B)**: KoBART (125M) 대비 100배 이상 큰 모델 사용
2. **4-bit 양자화**: 메모리 효율적으로 대형 모델 사용 가능
3. **Response-only Loss**: 프롬프트가 아닌 요약에만 집중하여 학습
4. **프롬프트 엔지니어링**: 화자 태그 유지, 길이 제한 등 명시적 지시
5. **MBR 앙상블**: 여러 프롬프트 변형의 예측을 합의하여 최종 선택

### 실패한 시도들

- **beam search**: Unsloth 4-bit 모델에서 지원되지 않음
- **sampling + reranking**: greedy 대비 유의미한 향상 없음
- **max_tokens 줄이기**: 요약이 잘리면서 ROUGE 하락
- **SimPO (선호 학습)**: SFT 대비 성능 하락

---

## 요약

이 노트북에서 다룬 내용:

1. **Unsloth**로 Qwen3-14B를 4-bit 양자화 로드
2. **LoRA** 어댑터 설정 (전체 파라미터의 0.86%만 학습)
3. **Response-only loss** 로 요약 생성에 집중 학습
4. **Dev set ROUGE 평가** (공백/Mecab 기반)
5. **Test set 제출 파일** 생성

### 더 높은 성능을 원한다면?

1. 여러 프롬프트 변형으로 추론 → **MBR 앙상블** (이 코드의 결과 + 프롬프트 변형 결과들을 합침)
2. 더 많은 에폭 학습 (단, 과적합 주의)
3. 데이터 증강 (대화 턴 일부 제거 등)